# OLAF — DocRED 5-document parallel smoke

Purpose: test **document-level parallelism with exactly 5 workers** before any larger OLAF run.

- Dataset: DocRED only
- Documents: first 5 records with exact `type == "dev"`
- `DOCUMENT_WORKERS = 5`
- One independent OLAF pipeline per process
- OpenRouter model: `openai/gpt-oss-20b`
- Reasoning effort: `minimal`
- Gold annotations are stripped before OLAF runs
- Each document writes its own result/debug files
- Existing completed outputs are resumed instead of paid again

The two OLAF LLM stages inside one document remain sequential. Therefore the intended peak is about
**5 concurrent OpenRouter requests**, not 10.


In [1]:
from __future__ import annotations

import os
import sys
import json
from pathlib import Path
from time import perf_counter
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()

BASE = HERE
while BASE.name != "olaf_baseline" and BASE.parent != BASE:
    BASE = BASE.parent
if BASE.name != "olaf_baseline":
    raise RuntimeError(
        "Run this notebook from inside the existing olaf_baseline folder."
    )

SRC = BASE / "src"
VENDOR = BASE / "vendor" / "olaf"
for p in [SRC, VENDOR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dotenv import load_dotenv
load_dotenv(BASE / ".env")

from dataset_io import (
    discover_ragtree_preprocessed,
    locate_dataset,
    read_jsonl,
    positive_gold_relation_count,
)
from parallel_docred5_runner import run_docred_worker

DOCUMENT_WORKERS = 5
MODEL = os.getenv("OLAF_OPENROUTER_MODEL", "openai/gpt-oss-20b")
REASONING_EFFORT = os.getenv("OLAF_REASONING_EFFORT", "minimal")
SPACY_MODEL = "en_core_web_sm"

print("BASE:", BASE)
print("Python:", sys.executable)
print("DOCUMENT_WORKERS:", DOCUMENT_WORKERS)
print("Model:", MODEL)

assert DOCUMENT_WORKERS == 5
assert "olaf_baseline" in str(sys.executable).lower()
assert ".venv" in str(sys.executable).lower()


c:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline
Python: c:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline\.venv\Scripts\python.exe
DOCUMENT_WORKERS: 5
Model: openai/gpt-oss-20b


In [2]:
# ZERO-COST local preflight.
import spacy
spacy.load(SPACY_MODEL)
print("spaCy model:", SPACY_MODEL, "OK")

if not os.getenv("OPENROUTER_API_KEY", "").strip():
    raise RuntimeError("OPENROUTER_API_KEY is missing.")

preprocessed = discover_ragtree_preprocessed(BASE)
docred_path = locate_dataset(preprocessed, "docred")
rows = read_jsonl(docred_path)

dev_rows = [row for row in rows if row.get("type") == "dev"]
selected = dev_rows[:5]

if len(selected) != 5:
    raise RuntimeError(f"Expected at least 5 exact type=dev records; found {len(dev_rows)}.")

print("DocRED file:", docred_path)
print("All rows:", len(rows))
print("Exact type=dev rows:", len(dev_rows))
print("\nSelected 5:")
selection_table = pd.DataFrame([
    {
        "index": i,
        "document_id": row.get("document_id"),
        "title": row.get("title"),
        "gold_relations_posthoc_only": positive_gold_relation_count(row),
        "text_chars": len(row.get("text", "")),
    }
    for i, row in enumerate(selected)
])
display(selection_table)

print("\nNo OLAF/OpenRouter calls made yet.")


spaCy model: en_core_web_sm OK
DocRED file: C:\Users\galencarmedeiro\RAGTree\preprocessed\docred_causal.jsonl
All rows: 106924
Exact type=dev rows: 998

Selected 5:


,index,document_id,title,gold_relations_posthoc_only,text_chars
0,0,DocRED - e37288ca6012859f,Skai TV,7,921
1,1,DocRED - 72971f0f36693b3a,Washington Place (West Virginia),17,801
2,2,DocRED - 0af496a208c087f2,IBM Research – Brazil,11,1422
3,3,DocRED - 3b611c48af52425e,Lookin Ass,6,807
4,4,DocRED - 2d3e5aaa2e9e3e9b,Conrad O. Johnson,21,1523



No OLAF/OpenRouter calls made yet.


## Paid parallel smoke

This cell launches the 5 selected DocRED documents simultaneously using the Windows `spawn` multiprocessing context.

Each worker has its own spaCy model, OLAF `Pipeline`, KR objects, OpenRouter adapter and debug log.


In [3]:
RUN_ROOT = BASE / "runs" / "docred_parallel5_smoke"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

started = perf_counter()
completed = []
failures = []

ctx = mp.get_context("spawn")

with ProcessPoolExecutor(
    max_workers=DOCUMENT_WORKERS,
    mp_context=ctx,
) as executor:
    future_to_row = {
        executor.submit(
            run_docred_worker,
            row,
            str(RUN_ROOT),
            MODEL,
            REASONING_EFFORT,
            SPACY_MODEL,
        ): row
        for row in selected
    }

    for future in as_completed(future_to_row):
        row = future_to_row[future]
        doc_id = row.get("document_id")
        try:
            result = future.result()
            completed.append(result)
            print(
                f"DONE | {doc_id} | "
                f"{result['pipeline_elapsed_seconds']:.2f}s | "
                f"concepts={result['concept_count']} | "
                f"relations={result['relation_count']} | "
                f"endpoint_relations={result['relations_with_both_endpoints']} | "
                f"resumed={result['resumed']}"
            )
        except Exception as exc:
            failures.append({
                "document_id": doc_id,
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
            print(f"FAILED | {doc_id} | {type(exc).__name__}: {exc}")

wall_seconds = perf_counter() - started

print("\nParallel invocation finished.")
print("Wall seconds:", round(wall_seconds, 3))
print("Completed:", len(completed))
print("Failed:", len(failures))


DONE | DocRED - e37288ca6012859f | 13.08s | concepts=1 | relations=7 | endpoint_relations=0 | resumed=False
DONE | DocRED - 72971f0f36693b3a | 17.98s | concepts=4 | relations=8 | endpoint_relations=0 | resumed=False
DONE | DocRED - 3b611c48af52425e | 20.38s | concepts=10 | relations=21 | endpoint_relations=10 | resumed=False
DONE | DocRED - 0af496a208c087f2 | 24.51s | concepts=13 | relations=20 | endpoint_relations=4 | resumed=False
DONE | DocRED - 2d3e5aaa2e9e3e9b | 433.44s | concepts=19 | relations=4 | endpoint_relations=0 | resumed=False

Parallel invocation finished.
Wall seconds: 460.312
Completed: 5
Failed: 0


In [4]:
# Offline summary.
completed_sorted = sorted(
    completed,
    key=lambda x: x["document_id"],
)

summary = pd.DataFrame([
    {
        "document_id": r["document_id"],
        "title": r["title"],
        "gold_relations_posthoc_only": r["gold_relation_count_posthoc_only"],
        "concepts": r["concept_count"],
        "relations": r["relation_count"],
        "relations_with_both_endpoints": r["relations_with_both_endpoints"],
        "pipeline_seconds": r["pipeline_elapsed_seconds"],
        "worker_wall_seconds": r["worker_wall_seconds"],
        "resumed": r["resumed"],
    }
    for r in completed_sorted
])

display(summary)

if not summary.empty:
    serial_equivalent = float(summary["pipeline_seconds"].sum())
    parallel_speedup = serial_equivalent / wall_seconds if wall_seconds else None

    print("\nCONCURRENCY")
    print("workers:", DOCUMENT_WORKERS)
    print("parallel wall seconds:", round(wall_seconds, 3))
    print("sum per-document pipeline seconds:", round(serial_equivalent, 3))
    print("observed wall-clock speedup vs summed pipeline time:",
          round(parallel_speedup, 3) if parallel_speedup else None)

    print("\nOUTPUT QUALITY DIAGNOSTIC")
    print("total concepts:", int(summary["concepts"].sum()))
    print("total native OLAF relations:", int(summary["relations"].sum()))
    print(
        "relations with both source+target endpoints:",
        int(summary["relations_with_both_endpoints"].sum()),
    )
    print(
        "posthoc gold relation count (diagnostic only):",
        int(summary["gold_relations_posthoc_only"].sum()),
    )

if failures:
    print("\nFAILURES")
    display(pd.DataFrame(failures))

print("\nRun root:", RUN_ROOT)


,document_id,title,gold_relations_posthoc_only,concepts,relations,relations_with_both_endpoints,pipeline_seconds,worker_wall_seconds,resumed
0,DocRED - 0af496a208c087f2,IBM Research – Brazil,11,13,20,4,24.514162,25.026768,False
1,DocRED - 2d3e5aaa2e9e3e9b,Conrad O. Johnson,21,19,4,0,433.435347,433.987135,False
2,DocRED - 3b611c48af52425e,Lookin Ass,6,10,21,10,20.380621,20.842621,False
3,DocRED - 72971f0f36693b3a,Washington Place (West Virginia),17,4,8,0,17.984733,18.476664,False
4,DocRED - e37288ca6012859f,Skai TV,7,1,7,0,13.082988,13.594129,False



CONCURRENCY
workers: 5
parallel wall seconds: 460.312
sum per-document pipeline seconds: 509.398
observed wall-clock speedup vs summed pipeline time: 1.107

OUTPUT QUALITY DIAGNOSTIC
total concepts: 47
total native OLAF relations: 60
relations with both source+target endpoints: 14
posthoc gold relation count (diagnostic only): 62

Run root: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline\runs\docred_parallel5_smoke


## Interpretation

This notebook tests **parallel execution**, not benchmark F1 yet.

Native OLAF relation labels/endpoints still require the small frozen projection adapter before they can be compared
fairly to DocRED's controlled relation IDs. Therefore the gold relation count shown above is diagnostic only and is
never given to OLAF.
